# OpenPlaque — RCA Adaptive Outer-Wall Plaque v1

Fresh-baseline developmental experiment. **Runtime → Run all**. The original successful RCA source geometry is retained; only the outer-wall model changes.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json, os, shutil, sys, time
DRIVE_ROOT = Path('/content/drive/MyDrive/OpenPlaque')
OUTPUT = DRIVE_ROOT / 'RCA_Adaptive_Outer_Wall_Plaque_v1'
REUSE_VALID_CACHES = True
FORCE_RECOMPUTE = False
if FORCE_RECOMPUTE and OUTPUT.exists(): shutil.rmtree(OUTPUT)
OUTPUT.mkdir(parents=True, exist_ok=True)
(OUTPUT/'notebook_started.json').write_text(json.dumps({'status':'started','time':time.time()}, indent=2))
print('Drive root:', DRIVE_ROOT)
print('Output:', OUTPUT)
print('Reuse valid caches:', REUSE_VALID_CACHES)


In [ ]:
import os, shutil, sys
os.chdir('/content')
REPO = Path('/content/OpenPlaque_rca_adaptive_outer_wall')
if REPO.exists(): shutil.rmtree(REPO)
BRANCH = 'rca-adaptive-outer-wall-plaque-from-main'
PINNED_SCIENCE_COMMIT = 'fe6891112c10d5d6ccddc69b0d4ffa18657ee504'
BASELINE = '0593b453959f5a353d644267fbeef24b514ef4d7'
print('Working directory repaired:', os.getcwd())
!git clone -q --branch $BRANCH https://github.com/pazzani/OpenPlaque.git $REPO
!git -C $REPO checkout -q $PINNED_SCIENCE_COMMIT
HEAD = get_ipython().getoutput(f'git -C {REPO} rev-parse HEAD')[0].strip()
MB = get_ipython().getoutput(f'git -C {REPO} merge-base HEAD {BASELINE}')[0].strip()
print('Checked out:', HEAD)
print('Merge base:', MB)
assert HEAD == PINNED_SCIENCE_COMMIT
assert MB == BASELINE
%pip install -q /content/OpenPlaque_rca_adaptive_outer_wall
for k in list(sys.modules):
    if k == 'openplaque' or k.startswith('openplaque.'):
        del sys.modules[k]
os.chdir('/content')


In [ ]:
from openplaque.rca_adaptive_outer_wall_plaque_v1 import synthetic_self_test
print('Synthetic self-test:', synthetic_self_test())
!pytest -q /content/OpenPlaque_rca_adaptive_outer_wall/tests/test_rca_adaptive_outer_wall_plaque_v1.py


In [ ]:
required = [
    DRIVE_ROOT/'Cache/Secondary_3D_Vesselness_Topology_v1/series7_int16.npy',
    DRIVE_ROOT/'Cache/Secondary_3D_Vesselness_Topology_v1/series7_int16.json',
    DRIVE_ROOT/'Cache/Source_Volume_Coronary_Centerlines/RCA_source_centerline.csv',
    DRIVE_ROOT/'Cache/Master_Coronary_Anatomy_Baseline_v2/master_anatomy_summary.json',
    DRIVE_ROOT/'Longitudinal_Plaque_PCAT_Fusion_v1/RCA_source_longitudinal_plaque_profile_1mm.csv',
    DRIVE_ROOT/'RCA_Source_Space_Plaque_Excess_Specificity_v1/summary.json',
]
missing = [str(p) for p in required if not p.exists()]
if missing: raise FileNotFoundError('Missing prerequisites:\n' + '\n'.join(missing))
(OUTPUT/'preflight_complete.json').write_text(json.dumps({'status':'complete','science_commit':PINNED_SCIENCE_COMMIT,'baseline':BASELINE,'required_count':len(required)}, indent=2))
print('Preflight complete:', len(required), 'required artifacts found')


In [ ]:
from openplaque.rca_adaptive_outer_wall_plaque_v1 import run
try:
    result = run(drive_root=str(DRIVE_ROOT), output_dir=str(OUTPUT))
    print(json.dumps(result['summary'], indent=2, default=str))
    print('Report:', result['report'])
    print('ZIP:', result['zip'])
except Exception as e:
    (OUTPUT/'notebook_failure.json').write_text(json.dumps({'status':'FAILED','type':type(e).__name__,'message':str(e)}, indent=2))
    raise
